# Customer Radar — huấn luyện ViSoBERT v4 (mỉa mai + số sao + nhiều nguyên nhân)

Giữ lại cho tương thích v4. Với bản v5 có nguyên nhân gốc rễ cấp 3 (L3 `gia_dung`), dùng `02_train_visobert_v5.ipynb`.

In [ ]:
import glob, os, shutil, subprocess
src = os.path.dirname(glob.glob("/kaggle/input/**/nlp_service/train.py", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/nlp_service", dirs_exist_ok=True)
os.chdir("/kaggle/working/nlp_service")
print(open("data/real/manifest.json", encoding="utf-8").read())

In [ ]:
from huggingface_hub import snapshot_download
base = snapshot_download("uitnlp/visobert", local_dir="/kaggle/working/visobert")
print(base)

In [ ]:
train_files = ["data/real/train.jsonl"] + [p for p in ["data/domain_train.jsonl", "data/visfd_train.jsonl"] if os.path.exists(p)]
dev = "data/real/dev.jsonl" if sum(1 for _ in open("data/real/dev.jsonl", encoding="utf-8")) >= 50 else "data/gold_dev.jsonl"
cmd = ["python", "train.py", "--train", *train_files, "--dev", dev, "--exclude", "data/real/test.jsonl",
       "--base", "/kaggle/working/visobert", "--out", "/kaggle/working/ckpt-v4",
       "--epochs", "4", "--batch-size", "32", "--max-len", "160"]
subprocess.run(cmd, check=True)

In [ ]:
subprocess.run(["python", "eval_real.py", "--checkpoint", "/kaggle/working/ckpt-v4",
                "--test", "data/real/test.jsonl", "--out", "/kaggle/working/test_report.json"], check=True)

In [ ]:
subprocess.run(["pip", "install", "-q", "onnx", "onnxruntime"], check=True)
subprocess.run(["python", "deploy/export_onnx.py", "--checkpoint", "/kaggle/working/ckpt-v4", "--out", "/kaggle/working/onnx-v4"], check=True)
shutil.make_archive("/kaggle/working/ckpt-v4", "zip", "/kaggle/working/ckpt-v4")
shutil.make_archive("/kaggle/working/onnx-v4", "zip", "/kaggle/working/onnx-v4")
print(os.listdir("/kaggle/working"))